# ECG XAI — Complete Thesis Notebook
### Reliability of Post-hoc Explanations for Deep ECG Classification

**Emine Nur Kahraman** · Data Science MSc · Yeditepe University

Full pipeline in chronological order, fully self-contained (no external `.py`
imports). Sections: model → data → training → evaluation → XAI engine →
attribution analyses → stability → model comparison → figures.


## 1. Kurulum ve importlar

In [2]:
# Gerekirse: %pip install wfdb captum shap lime neurokit2
import os, ast, numpy as np, pandas as pd, wfdb, torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import roc_auc_score, f1_score, classification_report

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# >>> KENDİ YOLUNU YAZ <<<
%cd ~/ENK/TEZ
data_dir = "ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3/"

TARGET_CLASSES = ['NORM', 'MI', 'STTC', 'CD', 'HYP']
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)

device: cuda
/home/ekaftech/ENK/TEZ


## 2. Model mimarisi (ResNet18-1D)

In [3]:
# 1D ResNet için Residual Blok
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=7, stride=stride, padding=3, bias=False)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=7, stride=1, padding=3, bias=False)
        self.bn2 = nn.BatchNorm1d(out_channels)

        # Skip connection için boyutları eşleştir
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(out_channels)
            )

    def forward(self, x):
        residual = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out += self.shortcut(residual)  # Artık bağlantı
        out = self.relu(out)
        return out


# 1D ResNet Modeli
class ResNet1D(nn.Module):
    def __init__(self, block, num_blocks, num_classes=5):
        super(ResNet1D, self).__init__()
        self.in_channels = 64

        self.conv1 = nn.Conv1d(12, 64, kernel_size=15, stride=2, padding=7, bias=False)
        self.bn1 = nn.BatchNorm1d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool1d(kernel_size=3, stride=2, padding=1)

        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)

        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, block, out_channels, num_blocks, stride):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for s in strides:
            layers.append(block(self.in_channels, out_channels, s))
            self.in_channels = out_channels
        return nn.Sequential(*layers)

    def forward(self, x):
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.maxpool(out)

        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)

        out = self.avgpool(out)
        out = self.flatten(out)
        out = self.fc(out)
        return out


# ResNet18'e benzer bir yapı oluşturalım
def ResNet18_1D():
    return ResNet1D(ResidualBlock, [2, 2, 2, 2])


## 3. Dataset (augmentation sadece train'de)

In [4]:
class EcgDataset(Dataset):
    def __init__(self, meta_df, labels, data_dir, is_train=False):
        self.meta_df = meta_df
        self.labels = labels
        self.data_dir = data_dir
        self.is_train = is_train

    def __len__(self):
        return len(self.meta_df)

    def __getitem__(self, idx):
        record_path = os.path.join(self.data_dir, self.meta_df.iloc[idx]['filename_lr'])
        signal, _ = wfdb.rdsamp(record_path)
        signal = np.transpose(signal, (1, 0))          # (12, 1000)

        if self.is_train:
            signal = signal * (1 + np.random.uniform(-0.1, 0.1))      # ölçekleme
            signal = signal + np.random.normal(0, 0.01, signal.shape) # gürültü
            signal = np.roll(signal, np.random.randint(-50, 50), axis=1)  # kaydırma

        return (torch.tensor(signal, dtype=torch.float32),
                torch.tensor(self.labels[idx], dtype=torch.float32))

## 4. Patient-wise split — `strat_fold`

PTB-XL dokümantasyonu: aynı hastanın tüm kayıtları aynı fold'a atanmıştır.
Fold 9 ve 10 en az bir insan değerlendirmesinden geçtiği için etiket kalitesi
daha yüksektir; bu yüzden 9 validation, 10 test olarak kullanılır.

In [5]:
df  = pd.read_csv(os.path.join(data_dir, 'ptbxl_database.csv'), index_col='ecg_id')
scp = pd.read_csv(os.path.join(data_dir, 'scp_statements.csv'), index_col=0)

diag_map = scp[scp.diagnostic_class.isin(TARGET_CLASSES)]['diagnostic_class'].to_dict()

def to_superclasses(scp_codes_str):
    d = ast.literal_eval(scp_codes_str)
    return sorted({diag_map[c] for c, w in d.items() if c in diag_map and w >= 50})

df['labels'] = df.scp_codes.apply(to_superclasses)

train_df = df[df.strat_fold.isin(range(1, 9))].copy()
val_df   = df[df.strat_fold == 9].copy()
test_df  = df[df.strat_fold == 10].copy()

mlb = MultiLabelBinarizer(classes=TARGET_CLASSES)
y_train = mlb.fit_transform(train_df['labels'])
y_val   = mlb.transform(val_df['labels'])
y_test  = mlb.transform(test_df['labels'])

print(f"train {len(train_df):6d}   val {len(val_df):6d}   test {len(test_df):6d}")
print("\nSınıf dağılımı (kayıt sayısı):")
print(pd.DataFrame({'train': y_train.sum(0), 'val': y_val.sum(0), 'test': y_test.sum(0)},
                   index=TARGET_CLASSES))

# hasta sızıntısı kontrolü — kesişim boş olmalı
pt = {k: set(d.patient_id) for k, d in
      [('train', train_df), ('val', val_df), ('test', test_df)]}
print("\nOrtak hasta (olmamalı):",
      len(pt['train'] & pt['val']), len(pt['train'] & pt['test']), len(pt['val'] & pt['test']))

train  17418   val   2183   test   2198

Sınıf dağılımı (kayıt sayısı):
      train  val  test
NORM   7542  942   954
MI     3307  412   415
STTC   4062  510   506
CD     3900  495   496
HYP    1820  216   222

Ortak hasta (olmamalı): 0 0 0


## 5. DataLoader ve ağırlıklı loss

In [6]:
train_loader = DataLoader(EcgDataset(train_df, y_train, data_dir, is_train=True),
                          batch_size=32, shuffle=True)
val_loader   = DataLoader(EcgDataset(val_df,   y_val,   data_dir), batch_size=32)
test_loader  = DataLoader(EcgDataset(test_df,  y_test,  data_dir), batch_size=32)

pos_counts = y_train.sum(axis=0)
class_weights = len(y_train) / (len(TARGET_CLASSES) * pos_counts)
pos_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights)

print("pos_weight:", dict(zip(TARGET_CLASSES, np.round(class_weights, 2))))

pos_weight: {'NORM': 0.46, 'MI': 1.05, 'STTC': 0.86, 'CD': 0.89, 'HYP': 1.91}


## 6. Eğitim — 20 epoch, ReduceLROnPlateau, en iyi val AUC checkpoint

In [7]:
model     = ResNet18_1D().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',
                                                       factor=0.1, patience=3)
CKPT = 'ecg_optimized_patientwise.pth'


def evaluate(loader):
    """labels, probs, mean loss döndürür."""
    model.eval(); L, P, tot, n = [], [], 0.0, 0
    with torch.no_grad():
        for sig, lab in loader:
            sig, lab = sig.to(device), lab.to(device)
            logits = model(sig)
            tot += criterion(logits, lab).item() * sig.size(0); n += sig.size(0)
            L.append(lab.cpu().numpy())
            P.append(torch.sigmoid(logits).cpu().numpy())
    return np.concatenate(L), np.concatenate(P), tot / n


best_auc = 0.0
for epoch in range(20):
    model.train(); run = 0.0
    for sig, lab in train_loader:
        sig, lab = sig.to(device), lab.to(device)
        loss = criterion(model(sig), lab)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        run += loss.item() * sig.size(0)
    tr_loss = run / len(train_loader.dataset)

    Lv, Pv, val_loss = evaluate(val_loader)
    val_auc = roc_auc_score(Lv, Pv, average='macro')
    scheduler.step(val_loss)

    tag = ""
    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model.state_dict(), CKPT)
        tag = "   *** best saved ***"
    print(f"Epoch [{epoch+1:2d}/20]  TrLoss {tr_loss:.4f}  "
          f"ValLoss {val_loss:.4f}  ValAUC {val_auc:.4f}{tag}")

print(f"\nEn iyi val AUC: {best_auc:.4f}  ->  {CKPT}")

Epoch [ 1/20]  TrLoss 0.3456  ValLoss 0.3177  ValAUC 0.8860   *** best saved ***
Epoch [ 2/20]  TrLoss 0.2962  ValLoss 0.3022  ValAUC 0.8982   *** best saved ***
Epoch [ 3/20]  TrLoss 0.2776  ValLoss 0.3225  ValAUC 0.9025   *** best saved ***
Epoch [ 4/20]  TrLoss 0.2653  ValLoss 0.3212  ValAUC 0.9062   *** best saved ***
Epoch [ 5/20]  TrLoss 0.2581  ValLoss 0.2719  ValAUC 0.9220   *** best saved ***
Epoch [ 6/20]  TrLoss 0.2518  ValLoss 0.3172  ValAUC 0.9035
Epoch [ 7/20]  TrLoss 0.2463  ValLoss 0.2885  ValAUC 0.9179
Epoch [ 8/20]  TrLoss 0.2403  ValLoss 0.2735  ValAUC 0.9221   *** best saved ***
Epoch [ 9/20]  TrLoss 0.2368  ValLoss 0.2859  ValAUC 0.9164
Epoch [10/20]  TrLoss 0.2171  ValLoss 0.2443  ValAUC 0.9327   *** best saved ***
Epoch [11/20]  TrLoss 0.2107  ValLoss 0.2494  ValAUC 0.9327   *** best saved ***
Epoch [12/20]  TrLoss 0.2092  ValLoss 0.2534  ValAUC 0.9311
Epoch [13/20]  TrLoss 0.2068  ValLoss 0.2470  ValAUC 0.9329   *** best saved ***
Epoch [14/20]  TrLoss 0.2039  V

## 7. Eşik optimizasyonu (val) + final rapor (test = fold 10)

In [8]:
model.load_state_dict(torch.load(CKPT, map_location=device))

Lv, Pv, _ = evaluate(val_loader)
thresholds = []
for i in range(len(TARGET_CLASSES)):
    best_f1, best_t = 0.0, 0.5
    for t in np.arange(0.1, 0.9, 0.01):
        f = f1_score(Lv[:, i], (Pv[:, i] > t).astype(int), zero_division=0)
        if f > best_f1:
            best_f1, best_t = f, t
    thresholds.append(best_t)
thresholds = np.array(thresholds)
print("Optimal eşikler (val):", dict(zip(TARGET_CLASSES, np.round(thresholds, 2))))

Lt, Pt, _ = evaluate(test_loader)
print(f"\n=== TEST (fold 10) — macro AUC: {roc_auc_score(Lt, Pt, average='macro'):.4f} ===")
print("\n--- Eşik 0.5 ---")
print(classification_report(Lt, (Pt > 0.5).astype(int),
                            target_names=TARGET_CLASSES, zero_division=0))
print("--- Optimize edilmiş eşikler ---")
print(classification_report(Lt, (Pt > thresholds).astype(int),
                            target_names=TARGET_CLASSES, zero_division=0))

# Aynı raporu validation için de al (tez tablosunda karşılaştırma için)
print("\n=== VALIDATION (fold 9) ===")
print(classification_report(Lv, (Pv > thresholds).astype(int),
                            target_names=TARGET_CLASSES, zero_division=0))

Optimal eşikler (val): {'NORM': 0.42, 'MI': 0.38, 'STTC': 0.35, 'CD': 0.41, 'HYP': 0.57}

=== TEST (fold 10) — macro AUC: 0.9301 ===

--- Eşik 0.5 ---
              precision    recall  f1-score   support

        NORM       0.85      0.84      0.85       954
          MI       0.76      0.63      0.69       415
        STTC       0.79      0.73      0.76       506
          CD       0.82      0.68      0.74       496
         HYP       0.64      0.57      0.60       222

   micro avg       0.80      0.73      0.77      2593
   macro avg       0.77      0.69      0.73      2593
weighted avg       0.80      0.73      0.76      2593
 samples avg       0.71      0.70      0.69      2593

--- Optimize edilmiş eşikler ---
              precision    recall  f1-score   support

        NORM       0.83      0.89      0.86       954
          MI       0.72      0.69      0.71       415
        STTC       0.72      0.82      0.77       506
          CD       0.79      0.72      0.75       496
  

---
## 8. XAI Engine

Defines the full explainability engine: Integrated Gradients, Grad-CAM (ReLU and
signed), GradientSHAP, LIME, the NeuroKit2 wave-overlap framework,
insertion/deletion faithfulness, causal wave ablation, and the Grad-CAM collapse
scan. Also exposes `main()` / `main_large()` that run the whole four-axis
evaluation. *(Previously imported as `xai_pipeline_v3`; inlined here.)*

Requires: `neurokit2`, `captum`, `shap` → `%pip install neurokit2 captum shap -q`

In [ ]:
"""
XAI Pipeline v3 — patient-wise model, fold 10 (held-out test)
==============================================================
v2'nin tüm bileşenleri, yeni `ecg_optimized_patientwise.pth` modeline bağlı.

Bileşenler:
  1. signed Grad-CAM  (ReLU yok, |cam|, min-max)   -> çöküş sorunu yok
  2. Integrated Gradients (captum, zero baseline, n_steps=50, sigmoid)
  3. GradientSHAP     (shap.GradientExplainer)
  4. LIME             (LimeTabularExplainer, sigmoid predict_fn)
  5. Wave-overlap     (NeuroKit2 dwt, Lead II, 75. persentil)
  6. Faithfulness     (insertion/deletion, n=20, median baseline, trapz)
  7. Causal ablation  (comprehensiveness / sufficiency, Monte Carlo N=30)
  8. Grad-CAM çöküş taraması (fold 10 üzerinde, sınıf bazlı oran)

Determinizm: NeuroKit2 dwt deterministik; ablation'daki Monte Carlo
gürültüsü SEED ile sabitlenir.

Kullanım:
    python xai_pipeline_v3.py
veya notebook'a hücre olarak yapıştır (main() çağır).
"""

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import neurokit2 as nk
import wfdb
from collections import defaultdict

# =============================================================================
# CONFIG
# =============================================================================
DATA_DIR   = "ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3/"
MODEL_PATH = "ecg_optimized_patientwise.pth"
OUT_DIR    = "xai_output_v3/"

CLASS_NAMES = ['NORM', 'MI', 'STTC', 'CD', 'HYP']
ANALYSIS_LEAD = 1        # Lead II
SR   = 100               # Hz
PCT  = 75                # yüksek-aktivasyon persentili
N_ID_STEPS = 20          # insertion/deletion adım sayısı
N_MC = 30                # ablation Monte Carlo tekrar sayısı
SEED = 42

# 2. anketteki 20 ECG (ptbxl_database.csv satır indeksi) — hepsi fold 10'da
SELECTION = [
    (1, 'NORM', 2927), (2, 'NORM', 14133), (3, 'NORM', 7769), (4, 'NORM', 15678),
    (5, 'MI',    955), (6, 'MI',   13810), (7, 'MI',   4155), (8, 'MI',    3107),
    (9, 'STTC',11105), (10,'STTC', 18386), (11,'STTC', 1611), (12,'STTC', 17102),
    (13,'CD',   1207), (14,'CD',   17501), (15,'CD',  15064), (16,'CD',    1043),
    (17,'HYP', 15749), (18,'HYP',   1772), (19,'HYP', 12160), (20,'HYP',   9000),
]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(SEED); torch.manual_seed(SEED)


# =============================================================================
# MODEL (v3 notebook ile aynı tanım)
# =============================================================================
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, 7, stride, 3, bias=False)
        self.bn1   = nn.BatchNorm1d(out_channels)
        self.relu  = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv1d(out_channels, out_channels, 7, 1, 3, bias=False)
        self.bn2   = nn.BatchNorm1d(out_channels)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, 1, stride, bias=False),
                nn.BatchNorm1d(out_channels))

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return self.relu(out)


class ResNet1D(nn.Module):
    def __init__(self, block, num_blocks, num_classes=5):
        super(ResNet1D, self).__init__()
        self.in_channels = 64
        self.conv1 = nn.Conv1d(12, 64, 15, 2, 7, bias=False)
        self.bn1 = nn.BatchNorm1d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool1d(3, 2, 1)
        self.layer1 = self._make_layer(block, 64,  num_blocks[0], 1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], 2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], 2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], 2)
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, block, out_channels, num_blocks, stride):
        layers = []
        for s in [stride] + [1] * (num_blocks - 1):
            layers.append(block(self.in_channels, out_channels, s))
            self.in_channels = out_channels
        return nn.Sequential(*layers)

    def forward(self, x):
        out = self.maxpool(self.relu(self.bn1(self.conv1(x))))
        out = self.layer4(self.layer3(self.layer2(self.layer1(out))))
        return self.fc(self.flatten(self.avgpool(out)))


def ResNet18_1D():
    return ResNet1D(ResidualBlock, [2, 2, 2, 2])


# =============================================================================
# SIGNED GRAD-CAM  (ReLU yok -> çöküş yok)
# =============================================================================
class GradCAMsigned1D:
    def __init__(self, model, target_layer):
        self.model = model
        self.a = None
        self.g = None
        self.fh = target_layer.register_forward_hook(
            lambda m, i, o: setattr(self, 'a', o.detach()))
        self.bh = target_layer.register_full_backward_hook(
            lambda m, gi, go: setattr(self, 'g', go[0].detach()))

    def __call__(self, x, class_idx=None):
        self.model.eval()
        out = self.model(x)
        probs = torch.sigmoid(out).detach().cpu().numpy()[0]
        if class_idx is None:
            class_idx = int(out.argmax(1).item())
        self.model.zero_grad()
        out[0, class_idx].backward(retain_graph=True)
        w = self.g.mean(dim=2, keepdim=True)
        cam = (w * self.a).sum(dim=1, keepdim=True)          # ReLU YOK
        cam = F.interpolate(cam, size=x.shape[2], mode='linear', align_corners=False)
        cam = np.abs(cam.squeeze().cpu().numpy())
        rng = cam.max() - cam.min()
        if rng > 0:
            cam = (cam - cam.min()) / rng
        return cam, class_idx, probs

    def remove(self):
        self.fh.remove(); self.bh.remove()


class GradCAMrelu1D(GradCAMsigned1D):
    """Orijinal (ReLU'lu) Grad-CAM — sadece çöküş oranını ölçmek için."""
    def __call__(self, x, class_idx=None):
        self.model.eval()
        out = self.model(x)
        probs = torch.sigmoid(out).detach().cpu().numpy()[0]
        if class_idx is None:
            class_idx = int(out.argmax(1).item())
        self.model.zero_grad()
        out[0, class_idx].backward(retain_graph=True)
        w = self.g.mean(dim=2, keepdim=True)
        cam = F.relu((w * self.a).sum(dim=1, keepdim=True))   # ReLU VAR
        cam = F.interpolate(cam, size=x.shape[2], mode='linear', align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        if cam.max() > 0:
            cam = (cam - cam.min()) / (cam.max() - cam.min())
        return cam, class_idx, probs


# =============================================================================
# DALGA TESPİTİ + OVERLAP
# =============================================================================
def detect_waves(signal_1d, sr=SR):
    try:
        cleaned = nk.ecg_clean(signal_1d, sampling_rate=sr)
        _, rpeaks = nk.ecg_peaks(cleaned, sampling_rate=sr)
        if len(rpeaks['ECG_R_Peaks']) < 2:
            return None
        _, waves = nk.ecg_delineate(cleaned, rpeaks, sampling_rate=sr, method="dwt")
        return waves
    except Exception:
        return None


def get_regions(waves, length=1000):
    R = {k: np.zeros(length, dtype=bool) for k in ['P', 'QRS', 'T', 'ST']}
    if waves is None:
        return R

    def fill(onsets, offsets, key):
        for on, off in zip(onsets, offsets):
            if not np.isnan(on) and not np.isnan(off):
                on, off = int(on), int(off)
                if 0 <= on < length and 0 <= off < length and on <= off:
                    R[key][on:off + 1] = True

    fill(waves.get('ECG_P_Onsets', []), waves.get('ECG_P_Offsets', []), 'P')
    fill(waves.get('ECG_R_Onsets', []), waves.get('ECG_R_Offsets', []), 'QRS')
    fill(waves.get('ECG_T_Onsets', []), waves.get('ECG_T_Offsets', []), 'T')
    fill(waves.get('ECG_R_Offsets', []), waves.get('ECG_T_Onsets', []), 'ST')   # ST
    return R


def calc_overlap(attr_1d, regions, pct=PCT):
    """Sıfır-varyans (çökmüş) haritada NaN döner — kirli değer üretmez."""
    if np.std(attr_1d) == 0:
        return {k: float('nan') for k in regions}
    thr = np.percentile(np.abs(attr_1d), pct)
    high = np.abs(attr_1d) >= thr
    total = high.sum()
    if total == 0:
        return {k: 0.0 for k in regions}
    return {k: float((high & v).sum() / total) for k, v in regions.items()}


# =============================================================================
# ATIF YÖNTEMLERİ — hepsi (1000,) temporal profil döndürür
# =============================================================================
def attr_signed(model, signal, cls):
    gc = GradCAMsigned1D(model, model.layer4[-1].conv2)
    cam, _, _ = gc(torch.FloatTensor(signal).unsqueeze(0).to(device), class_idx=cls)
    gc.remove()
    return cam


def attr_ig(model, signal, cls):
    from captum.attr import IntegratedGradients
    x = torch.FloatTensor(signal).unsqueeze(0).to(device).requires_grad_(True)
    ig = IntegratedGradients(lambda inp: torch.sigmoid(model(inp))[:, cls])
    a = ig.attribute(x, baselines=torch.zeros_like(x), n_steps=50,
                     method="gausslegendre")
    return np.abs(a.squeeze(0).detach().cpu().numpy()).sum(0)


def make_shap_explainer(model, background):
    import shap
    return shap.GradientExplainer(model, background)


def attr_shap(explainer, signal, cls):
    x = torch.FloatTensor(signal).unsqueeze(0).to(device)
    sv = explainer.shap_values(x)
    arr = sv[cls] if isinstance(sv, list) else sv[..., cls]
    return np.abs(np.asarray(arr).squeeze()).sum(0)


def attr_lime(model, signal, cls, train_flat, num_samples=500, seed=SEED):
    """
    Tabular LIME. Hiz icin: feature_selection='none' (ileri secim kapali),
    discretize_continuous=False, ikili sigmoid predict_fn (softmax DEGIL).
    """
    from lime.lime_tabular import LimeTabularExplainer

    def predict_fn(flat):
        out = []
        for i in range(0, len(flat), 64):
            x = torch.FloatTensor(flat[i:i+64].reshape(-1, 12, 1000)).to(device)
            with torch.no_grad():
                p = torch.sigmoid(model(x))[:, cls].cpu().numpy()
            out.append(p)
        p = np.concatenate(out)
        return np.stack([1 - p, p], axis=1)

    expl = LimeTabularExplainer(train_flat, mode='classification',
                                discretize_continuous=False,
                                feature_selection='none', random_state=seed)
    exp = expl.explain_instance(signal.flatten(), predict_fn,
                                num_features=12000, num_samples=num_samples,
                                labels=(1,))
    w = np.zeros(12000)
    for fid, val in exp.local_exp[1]:
        w[fid] = val
    return np.abs(w.reshape(12, 1000)).sum(0)


# =============================================================================
# FAITHFULNESS — insertion / deletion
# =============================================================================
def model_prob(model, sig, cls):
    with torch.no_grad():
        x = torch.FloatTensor(sig).unsqueeze(0).to(device)
        return torch.sigmoid(model(x))[0, cls].item()


def insertion_deletion(model, signal, attr_t, cls, n_steps=N_ID_STEPS):
    L = signal.shape[1]
    base = np.median(signal, axis=1, keepdims=True)      # izoelektrik
    order = np.argsort(-np.abs(attr_t))
    fracs = np.linspace(0, 1, n_steps + 1)
    base_full = np.repeat(base, L, axis=1)
    ins, dele = [], []
    for f in fracs:
        k = int(f * L)
        top = order[:k]
        d = signal.copy()
        s = base_full.copy()
        if k:
            d[:, top] = base
            s[:, top] = signal[:, top]
        dele.append(model_prob(model, d, cls))
        ins.append(model_prob(model, s, cls))
    return float(np.trapz(ins, fracs)), float(np.trapz(dele, fracs))


# =============================================================================
# NEDENSEL ABLATION — comprehensiveness / sufficiency, Monte Carlo
# =============================================================================
def wave_causal_mc(model, signal, class_idx=None, n_mc=N_MC, rng=None):
    """
    comp = p0 - p(dalga silinmiş)     yüksek -> dalga gerekli
    suff = p(yalnız dalga bırakılmış) yüksek -> dalga tek başına yeterli
    Baseline: per-lead medyan + per-lead std ile ölçeklenmiş Gauss gürültü.
    """
    if rng is None:
        rng = np.random.default_rng(SEED)
    model.eval()
    with torch.no_grad():
        out = model(torch.FloatTensor(signal).unsqueeze(0).to(device))
        cls = int(out.argmax(1).item()) if class_idx is None else class_idx
        p0 = torch.sigmoid(out)[0, cls].item()

    med = np.median(signal, axis=1, keepdims=True)
    std = np.std(signal, axis=1, keepdims=True)
    regions = get_regions(detect_waves(signal[ANALYSIS_LEAD]))

    res = {}
    for name, mask in regions.items():
        if not mask.any():
            res[name] = {'comp': float('nan'), 'comp_sd': float('nan'),
                         'suff': float('nan'), 'suff_sd': float('nan')}
            continue
        comps, suffs = [], []
        for _ in range(n_mc):
            noise = rng.normal(0, 1, signal.shape) * std + med
            d = signal.copy(); d[:, mask] = noise[:, mask]          # sil
            k = signal.copy(); k[:, ~mask] = noise[:, ~mask]        # yalnız tut
            comps.append(p0 - model_prob(model, d, cls))
            suffs.append(model_prob(model, k, cls))
        res[name] = {'comp': float(np.mean(comps)), 'comp_sd': float(np.std(comps)),
                     'suff': float(np.mean(suffs)), 'suff_sd': float(np.std(suffs))}
    return cls, p0, res


# =============================================================================
# GRAD-CAM ÇÖKÜŞ TARAMASI (fold 10)
# =============================================================================
def collapse_scan(model, df_fold, data_dir=None):
    data_dir = DATA_DIR if data_dir is None else data_dir
    gc = GradCAMrelu1D(model, model.layer4[-1].conv2)
    zero, total = defaultdict(int), defaultdict(int)
    for _, row in df_fold.iterrows():
        sig = load_signal(row['filename_lr'], data_dir)
        cam, pred, _ = gc(torch.FloatTensor(sig).unsqueeze(0).to(device))
        c = CLASS_NAMES[pred]
        total[c] += 1
        if np.std(cam) == 0:
            zero[c] += 1
    gc.remove()
    rows = []
    for c in CLASS_NAMES:
        n = total[c]
        rows.append({'class': c, 'zero': zero[c], 'total': n,
                     'rate': round(zero[c] / n, 4) if n else float('nan')})
    z, t = sum(zero.values()), sum(total.values())
    rows.append({'class': 'ALL', 'zero': z, 'total': t,
                 'rate': round(z / t, 4) if t else float('nan')})
    return pd.DataFrame(rows)


# =============================================================================
# YARDIMCI
# =============================================================================
def load_signal(filename_lr, data_dir=None):
    data_dir = DATA_DIR if data_dir is None else data_dir
    sig, _ = wfdb.rdsamp(os.path.join(data_dir, filename_lr))
    return np.transpose(sig, (1, 0)).astype(np.float32)     # (12, 1000)


# =============================================================================
# MAIN
# =============================================================================
def main(data_dir=None, model_path=None, out_dir=None,
         run_lime=False, run_shap=True, run_collapse=True):
    global DATA_DIR, MODEL_PATH, OUT_DIR
    if data_dir   is not None: DATA_DIR   = data_dir
    if model_path is not None: MODEL_PATH = model_path
    if out_dir    is not None: OUT_DIR    = out_dir
    os.makedirs(OUT_DIR, exist_ok=True)
    print("=" * 70)
    print("XAI PIPELINE v3 — patient-wise model, fold 10")
    print("=" * 70)

    # --- model ---
    model = ResNet18_1D().to(device)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    model.eval()
    print(f"[1] Model yüklendi: {MODEL_PATH}")

    # --- metadata ---
    df = pd.read_csv(os.path.join(DATA_DIR, 'ptbxl_database.csv'))
    fold10 = df[df.strat_fold == 10]
    print(f"[2] fold 10: {len(fold10)} kayıt")

    # --- SHAP background: fold 10'dan 50 örnek ---
    explainer = None
    if run_shap:
        bg_idx = fold10.sample(50, random_state=SEED).index
        bg = np.stack([load_signal(df.loc[i, 'filename_lr']) for i in bg_idx])
        explainer = make_shap_explainer(model, torch.FloatTensor(bg).to(device))
        print("[3] SHAP background hazır (fold 10, n=50)")

    # --- LIME için train dağılımı (flatten) ---
    train_flat = None
    if run_lime:
        tr_idx = df[df.strat_fold.isin(range(1, 9))].sample(200, random_state=SEED).index
        train_flat = np.stack([load_signal(df.loc[i, 'filename_lr']).flatten()
                               for i in tr_idx])
        print("[4] LIME referans dağılımı hazır (train, n=200)")

    # --- ana döngü: 20 ECG ---
    print("\n[5] 20 ECG analiz ediliyor...\n")
    rows = []
    ov_agg   = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))
    faith_agg = defaultdict(lambda: defaultdict(list))
    comp_agg = defaultdict(lambda: defaultdict(list))
    suff_agg = defaultdict(lambda: defaultdict(list))
    rng = np.random.default_rng(SEED)

    for n, true_cls, idx in SELECTION:
        sig = load_signal(df.loc[idx, 'filename_lr'])
        with torch.no_grad():
            probs = torch.sigmoid(
                model(torch.FloatTensor(sig).unsqueeze(0).to(device)))[0].cpu().numpy()
        cls = int(probs.argmax())
        pred = CLASS_NAMES[cls]
        correct = (pred == true_cls)

        regions = get_regions(detect_waves(sig[ANALYSIS_LEAD]))

        attrs = {'signed-GC': attr_signed(model, sig, cls),
                 'IG':        attr_ig(model, sig, cls)}
        if run_shap:
            attrs['SHAP'] = attr_shap(explainer, sig, cls)
        if run_lime:
            attrs['LIME'] = attr_lime(model, sig, cls, train_flat)

        row = {'ecg': n, 'true': true_cls, 'pred': pred,
               'prob': round(float(probs[cls]), 3), 'correct': correct}

        for mname, a in attrs.items():
            ov = calc_overlap(a, regions)
            ai, ad = insertion_deletion(model, sig, a, cls)
            row.update({f'{mname}_{w}': round(ov[w], 3) for w in ['P', 'QRS', 'ST', 'T']})
            row[f'{mname}_faith'] = round(ai - ad, 3)
            if correct:                                   # agregalar sadece doğru tahminler
                for w in ['P', 'QRS', 'ST', 'T']:
                    if not np.isnan(ov[w]):
                        ov_agg[mname][true_cls][w].append(ov[w])
                faith_agg[mname][true_cls].append(ai - ad)

        _, p0, cau = wave_causal_mc(model, sig, class_idx=cls, rng=rng)
        for w in ['P', 'QRS', 'ST', 'T']:
            row[f'abl_comp_{w}'] = round(cau[w]['comp'], 3)
            row[f'abl_suff_{w}'] = round(cau[w]['suff'], 3)
            if correct and not np.isnan(cau[w]['comp']):
                comp_agg[true_cls][w].append(cau[w]['comp'])
                suff_agg[true_cls][w].append(cau[w]['suff'])

        rows.append(row)
        mark = "OK " if correct else "XX "
        print(f"  {mark}#{n:2d} {true_cls:4s} -> {pred:4s} ({probs[cls]:.2f})")

    per_ecg = pd.DataFrame(rows)
    per_ecg.to_csv(os.path.join(OUT_DIR, 'per_ecg.csv'), index=False)

    # --- sınıf bazlı tablolar ---
    def dump(agg, fname, title, keys=('P', 'QRS', 'ST', 'T')):
        out = []
        for mname in agg:
            for c in CLASS_NAMES:
                d = {'method': mname, 'class': c}
                for w in keys:
                    v = agg[mname][c][w] if isinstance(agg[mname][c], dict) else None
                    d[w] = round(float(np.mean(v)), 3) if v else float('nan')
                out.append(d)
        t = pd.DataFrame(out)
        t.to_csv(os.path.join(OUT_DIR, fname), index=False)
        print(f"\n--- {title} ---")
        print(t.to_string(index=False))
        return t

    dump(ov_agg, 'overlap_per_class.csv', 'Wave-overlap (sınıf x yöntem)')

    frows = []
    for mname in faith_agg:
        for c in CLASS_NAMES:
            v = faith_agg[mname][c]
            frows.append({'method': mname, 'class': c,
                          'faith': round(float(np.mean(v)), 3) if v else float('nan')})
    ftab = pd.DataFrame(frows)
    ftab.to_csv(os.path.join(OUT_DIR, 'faithfulness_per_class.csv'), index=False)
    print("\n--- Faithfulness (ins - del) ---")
    print(ftab.pivot(index='class', columns='method', values='faith')
              .reindex(CLASS_NAMES).to_string())

    arows = []
    for c in CLASS_NAMES:
        d = {'class': c}
        for w in ['P', 'QRS', 'ST', 'T']:
            d[f'comp_{w}'] = round(float(np.mean(comp_agg[c][w])), 3) if comp_agg[c][w] else float('nan')
            d[f'suff_{w}'] = round(float(np.mean(suff_agg[c][w])), 3) if suff_agg[c][w] else float('nan')
        arows.append(d)
    atab = pd.DataFrame(arows)
    atab.to_csv(os.path.join(OUT_DIR, 'ablation_per_class.csv'), index=False)
    print("\n--- Causal ablation (comp = gerekli, suff = yeterli) ---")
    print(atab.to_string(index=False))

    # --- Grad-CAM çöküş taraması ---
    if run_collapse:
        print("\n[6] Grad-CAM (ReLU) çöküş taraması, fold 10...")
        ctab = collapse_scan(model, fold10)
        ctab.to_csv(os.path.join(OUT_DIR, 'gradcam_collapse.csv'), index=False)
        print(ctab.to_string(index=False))

    print(f"\nTüm çıktılar: {OUT_DIR}")
    return per_ecg


if __name__ == "__main__":
    main()


# =============================================================================
# BÜYÜK ÖRNEKLEM ANALİZİ — fold 10'dan sınıf başına N kayıt
# =============================================================================
def _fold10_labels(df):
    """fold 10 kayıtları için süpersınıf etiketlerini üretir."""
    import ast as _ast
    scp = pd.read_csv(os.path.join(DATA_DIR, 'scp_statements.csv'), index_col=0)
    dmap = scp[scp.diagnostic_class.isin(CLASS_NAMES)]['diagnostic_class'].to_dict()

    def to_super(s):
        d = _ast.literal_eval(s)
        return sorted({dmap[c] for c, w in d.items() if c in dmap and w >= 50})

    f10 = df[df.strat_fold == 10].copy()
    f10['labels'] = f10.scp_codes.apply(to_super)
    return f10


def main_large(n_per_class=40, min_prob=0.5, n_mc=10,
               data_dir=None, model_path=None, out_dir=None,
               run_shap=True, run_ablation=True, run_lime=False,
               lime_samples=500, seed=SEED):
    """
    fold 10'dan her sınıf için, modelin DOĞRU ve güvenle (>= min_prob)
    sınıflandırdığı n_per_class kayıt seçer; overlap + faithfulness
    (+ ablation) hesaplar. 20 anketlik örneklemdeki n<5 sorununu çözer.

    n_mc küçük tutulur (varsayılan 10) çünkü kayıt sayısı büyük;
    Monte Carlo gürültüsü kayıtlar arası ortalamada zaten sönümlenir.
    """
    global DATA_DIR, MODEL_PATH, OUT_DIR
    if data_dir   is not None: DATA_DIR   = data_dir
    if model_path is not None: MODEL_PATH = model_path
    OUT_DIR = out_dir if out_dir is not None else OUT_DIR
    os.makedirs(OUT_DIR, exist_ok=True)
    rng = np.random.default_rng(seed)

    print("=" * 70)
    print(f"BÜYÜK ÖRNEKLEM — fold 10, sınıf başına {n_per_class} kayıt")
    print("=" * 70)

    model = ResNet18_1D().to(device)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    model.eval()

    df = pd.read_csv(os.path.join(DATA_DIR, 'ptbxl_database.csv'))
    f10 = _fold10_labels(df)
    print(f"[1] fold 10: {len(f10)} kayıt")

    # --- tüm fold 10 için tahmin ---
    print("[2] Tahminler hesaplanıyor...")
    preds, probs_all = [], []
    for fn in f10['filename_lr']:
        s = load_signal(fn)
        with torch.no_grad():
            p = torch.sigmoid(
                model(torch.FloatTensor(s).unsqueeze(0).to(device)))[0].cpu().numpy()
        preds.append(int(p.argmax())); probs_all.append(float(p.max()))
    f10['pred'] = [CLASS_NAMES[i] for i in preds]
    f10['prob'] = probs_all

    # --- seçim: doğru + güvenli ---
    chosen = []
    for c in CLASS_NAMES:
        ok = f10[(f10['pred'] == c) & (f10['prob'] >= min_prob)
                 & (f10['labels'].apply(lambda L: c in L))]
        k = min(n_per_class, len(ok))
        if k < n_per_class:
            print(f"    UYARI: {c} için sadece {k} uygun kayıt var")
        pick = ok.sample(k, random_state=seed) if k else ok
        for _, r in pick.iterrows():
            chosen.append((c, r['filename_lr'], r['prob']))
    print(f"[3] Seçilen: {len(chosen)} kayıt "
          f"({dict(pd.Series([c for c,_,_ in chosen]).value_counts())})")

    # --- LIME referans dagilimi (train fold'undan) ---
    train_flat = None
    if run_lime:
        tr_idx = df[df.strat_fold.isin(range(1, 9))].sample(200, random_state=seed).index
        train_flat = np.stack([load_signal(df.loc[i, 'filename_lr']).flatten()
                               for i in tr_idx])
        print("[3b] LIME referans dagilimi hazir (train, n=200)")

    # --- SHAP background ---
    explainer = None
    if run_shap:
        bgn = f10.sample(50, random_state=seed)['filename_lr']
        bg = np.stack([load_signal(fn) for fn in bgn])
        explainer = make_shap_explainer(model, torch.FloatTensor(bg).to(device))
        print("[4] SHAP background hazır (n=50)")

    # --- analiz ---
    print("[5] Analiz...\n")
    rows = []
    for i, (true_cls, fn, prob) in enumerate(chosen):
        sig = load_signal(fn)
        cls = CLASS_NAMES.index(true_cls)
        regions = get_regions(detect_waves(sig[ANALYSIS_LEAD]))
        if not any(m.any() for m in regions.values()):
            continue                                   # dalga tespiti başarısız

        attrs = {'signed-GC': attr_signed(model, sig, cls),
                 'IG':        attr_ig(model, sig, cls)}
        if run_shap:
            attrs['SHAP'] = attr_shap(explainer, sig, cls)
        if run_lime:
            attrs['LIME'] = attr_lime(model, sig, cls, train_flat,
                                      num_samples=lime_samples, seed=seed)

        row = {'file': fn, 'true': true_cls, 'prob': round(prob, 3)}
        for m, a in attrs.items():
            ov = calc_overlap(a, regions)
            ai, ad = insertion_deletion(model, sig, a, cls)
            row.update({f'{m}_{w}': ov[w] for w in ['P', 'QRS', 'ST', 'T']})
            row[f'{m}_faith'] = ai - ad

        if run_ablation:
            _, _, cau = wave_causal_mc(model, sig, class_idx=cls, n_mc=n_mc, rng=rng)
            for w in ['P', 'QRS', 'ST', 'T']:
                row[f'comp_{w}'] = cau[w]['comp']
                row[f'suff_{w}'] = cau[w]['suff']

        rows.append(row)
        if (i + 1) % 20 == 0:
            print(f"    {i+1}/{len(chosen)}")

    d = pd.DataFrame(rows)
    d.to_csv(os.path.join(OUT_DIR, 'large_per_record.csv'), index=False)
    print(f"\n[6] Tamam: {len(d)} kayıt analiz edildi\n")

    # --- özet tablolar: ortalama ± standart sapma, n ---
    methods = [m for m in ['signed-GC', 'IG', 'SHAP', 'LIME'] if f'{m}_faith' in d.columns]

    print("=== WAVE-OVERLAP (ortalama, n) ===")
    ovr = []
    for m in methods:
        for c in CLASS_NAMES:
            sub = d[d.true == c]
            r = {'method': m, 'class': c, 'n': len(sub)}
            for w in ['P', 'QRS', 'ST', 'T']:
                r[w] = round(sub[f'{m}_{w}'].mean(), 3)
            ovr.append(r)
    ovt = pd.DataFrame(ovr); ovt.to_csv(os.path.join(OUT_DIR, 'large_overlap.csv'), index=False)
    print(ovt.to_string(index=False))

    if run_lime:
        mx = d[[c for c in d.columns if c.startswith('LIME_')
                and c.endswith(('P','QRS','ST','T'))]]
        print("\n=== LIME NOT: atif buyuklugu asagida raporlanan overlap'ten "
              "bagimsiz olarak cok kucuktur (bkz. large_per_record.csv) ===")

    print("\n=== FAITHFULNESS (ortalama ± sd) ===")
    fr = []
    for m in methods:
        for c in CLASS_NAMES:
            sub = d[d.true == c][f'{m}_faith']
            fr.append({'method': m, 'class': c, 'n': len(sub),
                       'faith': round(sub.mean(), 3), 'sd': round(sub.std(), 3)})
    ft = pd.DataFrame(fr); ft.to_csv(os.path.join(OUT_DIR, 'large_faithfulness.csv'), index=False)
    print(ft.to_string(index=False))

    if run_ablation:
        print("\n=== CAUSAL ABLATION (ortalama ± sd) ===")
        ar = []
        for c in CLASS_NAMES:
            sub = d[d.true == c]
            r = {'class': c, 'n': len(sub)}
            for w in ['P', 'QRS', 'ST', 'T']:
                r[f'comp_{w}'] = round(sub[f'comp_{w}'].mean(), 3)
                r[f'comp_{w}_sd'] = round(sub[f'comp_{w}'].std(), 3)
                r[f'suff_{w}'] = round(sub[f'suff_{w}'].mean(), 3)
            ar.append(r)
        at = pd.DataFrame(ar); at.to_csv(os.path.join(OUT_DIR, 'large_ablation.csv'), index=False)
        cols = ['class', 'n'] + [f'comp_{w}' for w in ['P','QRS','ST','T']] \
                              + [f'suff_{w}' for w in ['P','QRS','ST','T']]
        print(at[cols].to_string(index=False))

    print(f"\nÇıktılar: {OUT_DIR}")
    return d


# expose as `xai` for the calls below
import types as _t
xai = _t.ModuleType('xai')
xai.__dict__.update({k:v for k,v in globals().items() if not k.startswith('_')})
print('XAI engine ready — xai.main_large available')


---
## 9. Run the Four-Axis XAI Evaluation

Runs overlap, faithfulness, agreement and causal ablation on 40 correctly
classified, confident recordings per class. Produces the tables reported in the
thesis (`xai_output_v3/`).

In [ ]:
large = xai.main_large(
    n_per_class = 40,
    min_prob    = 0.5,
    n_mc        = 10,
    data_dir    = data_dir,
    model_path  = 'ecg_optimized_patientwise.pth',
    out_dir     = 'xai_output_v3/',
    run_shap    = True,
    run_ablation= True,
)

---
## 10. Stability Analysis — 5 Training Seeds

Retrains under 5 seeds and measures how much attributions change while
performance stays fixed. *(Previously `stability_v3`.)*

In [ ]:
"""
STABILITY ANALYSIS (Aşama 2) — 5 seed, patient-wise split
==========================================================
Danışman notu 2: tek seed (42) yerine 5 farklı seed ile aynı modeli sıfırdan
eğit, aynı EKG'ler için atıfları karşılaştır.

İki kararlılık ölçüsü:
  1. Sıralama kararlılığı  : Spearman ρ + top-k Jaccard (model çiftleri arası)
  2. Dalga-seviyesi kararlılık : wave-overlap skorunun 5 model arası std sapması

Tasarım kararı (önemli):
  Kayıt kümesi ve hedef sınıf 5 model için SABİT tutulur (gerçek etikete göre
  seçilir), çünkü "modelin doğru bildiği kayıtlar" modele göre değişir ve
  atıflar karşılaştırılamaz hale gelir.

Kullanım:
# st already defined in Section 10 (inlined)
    st.train_seeds(seeds=[0,1,2,3,4], data_dir=data_dir)      # ~5 x eğitim süresi
    res = st.analyze_stability(data_dir=data_dir, n_per_class=40)
"""

import os, ast
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import wfdb
import neurokit2 as nk
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr

# =============================================================================
# CONFIG
# =============================================================================
DATA_DIR = "ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3/"
OUT_DIR  = "stability_output/"
CKPT_FMT = "ecg_seed{seed}.pth"

CLASS_NAMES = ['NORM', 'MI', 'STTC', 'CD', 'HYP']
LEAD, SR, PCT = 1, 100, 75
TOPK = 50                      # top-k Jaccard için k
EPOCHS = 20

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# =============================================================================
# MODEL (v3 ile birebir aynı)
# =============================================================================
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, 7, stride, 3, bias=False)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv1d(out_channels, out_channels, 7, 1, 3, bias=False)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, 1, stride, bias=False),
                nn.BatchNorm1d(out_channels))

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return self.relu(out)


class ResNet1D(nn.Module):
    def __init__(self, block, num_blocks, num_classes=5):
        super(ResNet1D, self).__init__()
        self.in_channels = 64
        self.conv1 = nn.Conv1d(12, 64, 15, 2, 7, bias=False)
        self.bn1 = nn.BatchNorm1d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool1d(3, 2, 1)
        self.layer1 = self._make_layer(block, 64,  num_blocks[0], 1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], 2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], 2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], 2)
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, block, out_channels, num_blocks, stride):
        layers = []
        for s in [stride] + [1] * (num_blocks - 1):
            layers.append(block(self.in_channels, out_channels, s))
            self.in_channels = out_channels
        return nn.Sequential(*layers)

    def forward(self, x):
        out = self.maxpool(self.relu(self.bn1(self.conv1(x))))
        out = self.layer4(self.layer3(self.layer2(self.layer1(out))))
        return self.fc(self.flatten(self.avgpool(out)))


def ResNet18_1D():
    return ResNet1D(ResidualBlock, [2, 2, 2, 2])


# =============================================================================
# VERİ
# =============================================================================
class EcgDataset(Dataset):
    def __init__(self, meta_df, labels, data_dir, is_train=False):
        self.meta_df, self.labels = meta_df, labels
        self.data_dir, self.is_train = data_dir, is_train

    def __len__(self):
        return len(self.meta_df)

    def __getitem__(self, idx):
        p = os.path.join(self.data_dir, self.meta_df.iloc[idx]['filename_lr'])
        sig, _ = wfdb.rdsamp(p)
        sig = np.transpose(sig, (1, 0))
        if self.is_train:
            sig = sig * (1 + np.random.uniform(-0.1, 0.1))
            sig = sig + np.random.normal(0, 0.01, sig.shape)
            sig = np.roll(sig, np.random.randint(-50, 50), axis=1)
        return (torch.tensor(sig, dtype=torch.float32),
                torch.tensor(self.labels[idx], dtype=torch.float32))


def load_metadata(data_dir=None):
    data_dir = DATA_DIR if data_dir is None else data_dir
    df  = pd.read_csv(os.path.join(data_dir, 'ptbxl_database.csv'))
    scp = pd.read_csv(os.path.join(data_dir, 'scp_statements.csv'), index_col=0)
    dmap = scp[scp.diagnostic_class.isin(CLASS_NAMES)]['diagnostic_class'].to_dict()

    def to_super(s):
        d = ast.literal_eval(s)
        return sorted({dmap[c] for c, w in d.items() if c in dmap and w >= 50})

    df['labels'] = df.scp_codes.apply(to_super)
    return df


def load_signal(filename_lr, data_dir=None):
    data_dir = DATA_DIR if data_dir is None else data_dir
    sig, _ = wfdb.rdsamp(os.path.join(data_dir, filename_lr))
    return np.transpose(sig, (1, 0)).astype(np.float32)


# =============================================================================
# 1) 5 SEED İLE EĞİTİM
# =============================================================================
def train_seeds(seeds=(0, 1, 2, 3, 4), data_dir=None, epochs=EPOCHS, out_dir=None):
    """Her seed için sıfırdan eğitir; fold 1-8 train, 9 val, en iyi val AUC kaydedilir."""
    global DATA_DIR, OUT_DIR
    if data_dir is not None: DATA_DIR = data_dir
    if out_dir  is not None: OUT_DIR  = out_dir
    os.makedirs(OUT_DIR, exist_ok=True)

    df = load_metadata()
    tr = df[df.strat_fold.isin(range(1, 9))]
    va = df[df.strat_fold == 9]

    mlb = MultiLabelBinarizer(classes=CLASS_NAMES)
    y_tr = mlb.fit_transform(tr['labels'])
    y_va = mlb.transform(va['labels'])

    pos = y_tr.sum(axis=0)
    w = torch.tensor(len(y_tr) / (len(CLASS_NAMES) * pos), dtype=torch.float32).to(device)

    summary = []
    for seed in seeds:
        print(f"\n{'='*60}\nSEED {seed}\n{'='*60}")
        torch.manual_seed(seed); np.random.seed(seed)

        tl = DataLoader(EcgDataset(tr, y_tr, DATA_DIR, True), batch_size=32, shuffle=True)
        vl = DataLoader(EcgDataset(va, y_va, DATA_DIR), batch_size=32)

        model = ResNet18_1D().to(device)
        opt = torch.optim.Adam(model.parameters(), lr=1e-3)
        sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min',
                                                         factor=0.1, patience=3)
        crit = nn.BCEWithLogitsLoss(pos_weight=w)
        ckpt = os.path.join(OUT_DIR, CKPT_FMT.format(seed=seed))
        best = 0.0

        for ep in range(epochs):
            model.train()
            for sig, lab in tl:
                sig, lab = sig.to(device), lab.to(device)
                loss = crit(model(sig), lab)
                opt.zero_grad(); loss.backward(); opt.step()

            model.eval(); L, P, tot, n = [], [], 0.0, 0
            with torch.no_grad():
                for sig, lab in vl:
                    sig, lab = sig.to(device), lab.to(device)
                    lo = model(sig)
                    tot += crit(lo, lab).item() * sig.size(0); n += sig.size(0)
                    L.append(lab.cpu().numpy()); P.append(torch.sigmoid(lo).cpu().numpy())
            L, P = np.concatenate(L), np.concatenate(P)
            auc = roc_auc_score(L, P, average='macro')
            sch.step(tot / n)
            if auc > best:
                best = auc; torch.save(model.state_dict(), ckpt)
            print(f"  ep {ep+1:2d}/{epochs}  valAUC {auc:.4f}"
                  f"{'  *' if auc == best else ''}")

        print(f"  -> seed {seed}: en iyi val AUC {best:.4f}  ({ckpt})")
        summary.append({'seed': seed, 'best_val_auc': round(best, 4), 'ckpt': ckpt})

    s = pd.DataFrame(summary)
    s.to_csv(os.path.join(OUT_DIR, 'seed_summary.csv'), index=False)
    print("\n=== EĞİTİM ÖZETİ ===")
    print(s.to_string(index=False))
    print(f"\nAUC: {s.best_val_auc.mean():.4f} ± {s.best_val_auc.std():.4f}")
    return s


# =============================================================================
# 2) ATIF + KARARLILIK
# =============================================================================
def detect_waves(s, sr=SR):
    try:
        c = nk.ecg_clean(s, sampling_rate=sr)
        _, rp = nk.ecg_peaks(c, sampling_rate=sr)
        if len(rp['ECG_R_Peaks']) < 2:
            return None
        _, w = nk.ecg_delineate(c, rp, sampling_rate=sr, method="dwt")
        return w
    except Exception:
        return None


def get_regions(waves, length=1000):
    R = {k: np.zeros(length, bool) for k in ['P', 'QRS', 'T', 'ST']}
    if waves is None:
        return R

    def fill(on, off, key):
        for a, b in zip(on, off):
            if not np.isnan(a) and not np.isnan(b):
                a, b = int(a), int(b)
                if 0 <= a < length and 0 <= b < length and a <= b:
                    R[key][a:b + 1] = True

    fill(waves.get('ECG_P_Onsets', []), waves.get('ECG_P_Offsets', []), 'P')
    fill(waves.get('ECG_R_Onsets', []), waves.get('ECG_R_Offsets', []), 'QRS')
    fill(waves.get('ECG_T_Onsets', []), waves.get('ECG_T_Offsets', []), 'T')
    fill(waves.get('ECG_R_Offsets', []), waves.get('ECG_T_Onsets', []), 'ST')
    return R


def calc_overlap(attr, regions, pct=PCT):
    if np.std(attr) == 0:
        return {k: float('nan') for k in regions}
    thr = np.percentile(np.abs(attr), pct)
    high = np.abs(attr) >= thr
    tot = high.sum()
    if tot == 0:
        return {k: 0.0 for k in regions}
    return {k: float((high & v).sum() / tot) for k, v in regions.items()}


def attr_ig(model, sig, cls):
    from captum.attr import IntegratedGradients
    x = torch.FloatTensor(sig).unsqueeze(0).to(device).requires_grad_(True)
    ig = IntegratedGradients(lambda inp: torch.sigmoid(model(inp))[:, cls])
    a = ig.attribute(x, baselines=torch.zeros_like(x), n_steps=50,
                     method="gausslegendre")
    return np.abs(a.squeeze(0).detach().cpu().numpy()).sum(0)


def topk_jaccard(a, b, k=TOPK):
    ta = set(np.argsort(-np.abs(a))[:k].tolist())
    tb = set(np.argsort(-np.abs(b))[:k].tolist())
    return len(ta & tb) / len(ta | tb)


def analyze_stability(seeds=(0, 1, 2, 3, 4), n_per_class=40, data_dir=None,
                      out_dir=None, seed=42):
    """
    Sabit kayıt kümesi + sabit hedef sınıf üzerinde 5 modelin IG atıflarını
    karşılaştırır. Kayıtlar GERÇEK etikete göre seçilir (modele bağlı değil).
    """
    global DATA_DIR, OUT_DIR
    if data_dir is not None: DATA_DIR = data_dir
    if out_dir  is not None: OUT_DIR  = out_dir
    os.makedirs(OUT_DIR, exist_ok=True)

    df = load_metadata()
    f10 = df[df.strat_fold == 10]

    # --- sabit kayıt kümesi: her sınıftan n_per_class, gerçek etikete göre ---
    chosen = []
    for c in CLASS_NAMES:
        ok = f10[f10['labels'].apply(lambda L: c in L)]
        k = min(n_per_class, len(ok))
        for _, r in ok.sample(k, random_state=seed).iterrows():
            chosen.append((c, r['filename_lr']))
    print(f"[1] Sabit kayıt kümesi: {len(chosen)} "
          f"({dict(pd.Series([c for c, _ in chosen]).value_counts())})")

    # --- modelleri yükle ---
    models = []
    for s in seeds:
        p = os.path.join(OUT_DIR, CKPT_FMT.format(seed=s))
        m = ResNet18_1D().to(device)
        m.load_state_dict(torch.load(p, map_location=device))
        m.eval(); models.append(m)
    print(f"[2] {len(models)} model yüklendi")

    # --- her kayıt için 5 atıf + karşılaştırma ---
    print("[3] Atıflar hesaplanıyor...\n")
    rows = []
    for i, (true_cls, fn) in enumerate(chosen):
        sig = load_signal(fn)
        cls = CLASS_NAMES.index(true_cls)              # hedef sınıf SABİT
        regions = get_regions(detect_waves(sig[LEAD]))
        if not any(m.any() for m in regions.values()):
            continue

        attrs = [attr_ig(m, sig, cls) for m in models]
        ovs = [calc_overlap(a, regions) for a in attrs]

        # ikili karşılaştırmalar
        rhos, jacs = [], []
        for a in range(len(attrs)):
            for b in range(a + 1, len(attrs)):
                r = spearmanr(attrs[a], attrs[b]).correlation
                if not np.isnan(r):
                    rhos.append(r)
                jacs.append(topk_jaccard(attrs[a], attrs[b]))

        row = {'file': fn, 'true': true_cls,
               'spearman_mean': float(np.mean(rhos)) if rhos else float('nan'),
               'spearman_min': float(np.min(rhos)) if rhos else float('nan'),
               'jaccard_mean': float(np.mean(jacs))}
        # dalga-seviyesi kararlılık: 5 model arası std
        for w in ['P', 'QRS', 'ST', 'T']:
            vals = [o[w] for o in ovs if not np.isnan(o[w])]
            row[f'ov_{w}_mean'] = float(np.mean(vals)) if vals else float('nan')
            row[f'ov_{w}_sd']   = float(np.std(vals))  if vals else float('nan')
        rows.append(row)

        if (i + 1) % 20 == 0:
            print(f"    {i+1}/{len(chosen)}")

    d = pd.DataFrame(rows)
    d.to_csv(os.path.join(OUT_DIR, 'stability_per_record.csv'), index=False)

    # --- özet ---
    print(f"\n[4] {len(d)} kayıt analiz edildi\n")
    print("=== SIRALAMA KARARLILIĞI (model çiftleri arası) ===")
    t1 = d.groupby('true')[['spearman_mean', 'spearman_min', 'jaccard_mean']] \
          .mean().reindex(CLASS_NAMES).round(3)
    t1['n'] = d.groupby('true').size().reindex(CLASS_NAMES)
    print(t1.to_string())

    print("\n=== DALGA-SEVİYESİ KARARLILIK (5 model arası std) ===")
    cols = [f'ov_{w}_sd' for w in ['P', 'QRS', 'ST', 'T']]
    t2 = d.groupby('true')[cols].mean().reindex(CLASS_NAMES).round(3)
    print(t2.to_string())

    print("\n=== ORTALAMA OVERLAP (referans) ===")
    t3 = d.groupby('true')[[f'ov_{w}_mean' for w in ['P','QRS','ST','T']]] \
          .mean().reindex(CLASS_NAMES).round(3)
    print(t3.to_string())

    t1.to_csv(os.path.join(OUT_DIR, 'stability_ranking.csv'))
    t2.to_csv(os.path.join(OUT_DIR, 'stability_wave.csv'))
    print(f"\nÇıktılar: {OUT_DIR}")
    return d


import types as _t
st = _t.ModuleType('st')
st.__dict__.update({k:v for k,v in globals().items() if not k.startswith('_')})
res = st.analyze_stability(seeds=[0,1,2,3,4], n_per_class=40, data_dir=data_dir)
print('Stability analysis complete')


---
## 11. Model Comparison — Six Architectures

Compares CNN, ResNet, +weighted loss, +augmentation, Transformer and the
optimized ResNet under the same patient-wise split. *(Previously
`model_comparison`.)*

In [ ]:
"""
MODEL KARŞILAŞTIRMASI — patient-wise split
===========================================
Tez tablosundaki 6 modelin hepsini AYNI koşulda eğitir:
  fold 1-8 train, fold 9 validation, fold 10 test
  hepsinde best-checkpoint (en yüksek val AUC) — eski tabloda sadece
  optimized modelde vardı, bu haksız karşılaştırmaydı.

Modeller:
  1. CNN            Simple1DCNN,  5 epoch, düz BCE,      augmentation yok
  2. ResNet         ResNet18-1D, 10 epoch, düz BCE,      augmentation yok
  3. ResNet+weight  ResNet18-1D, 10 epoch, pos_weight,   augmentation yok
  4. ResNet+aug     ResNet18-1D, 15 epoch, pos_weight,   augmentation var
  5. Transformer    EcgTransformer, 15 epoch, pos_weight, augmentation var
  6. Optimized      ResNet18-1D, 20 epoch, pos_weight+aug+ReduceLROnPlateau
                    (varsayılan olarak ATLANIR; mevcut checkpoint kullanılsın,
                     XAI analizleri o ağırlıklara bağlı)

Kullanım:
# mc already defined in Section 11 (inlined)
    res = mc.compare_models(data_dir=data_dir)
"""

import os, ast, math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import wfdb
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import roc_auc_score, f1_score

DATA_DIR = "ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3/"
OUT_DIR  = "model_comparison_output/"
CLASS_NAMES = ['NORM', 'MI', 'STTC', 'CD', 'HYP']
SEED = 42
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# =============================================================================
# MİMARİLER (notebook'tan birebir)
# =============================================================================
class Simple1DCNN(nn.Module):
    def __init__(self, num_classes=5):
        super(Simple1DCNN, self).__init__()
        self.conv_block1 = nn.Sequential(
            nn.Conv1d(12, 32, kernel_size=7, stride=1, padding=3),
            nn.ReLU(), nn.MaxPool1d(2, 2))
        self.conv_block2 = nn.Sequential(
            nn.Conv1d(32, 64, kernel_size=5, stride=1, padding=2),
            nn.ReLU(), nn.MaxPool1d(2, 2))
        self.conv_block3 = nn.Sequential(
            nn.Conv1d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(), nn.MaxPool1d(2, 2))
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(128 * 125, 256)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.conv_block3(self.conv_block2(self.conv_block1(x)))
        return self.fc2(self.relu(self.fc1(self.flatten(x))))


class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, 7, stride, 3, bias=False)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv1d(out_channels, out_channels, 7, 1, 3, bias=False)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, 1, stride, bias=False),
                nn.BatchNorm1d(out_channels))

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return self.relu(out)


class ResNet1D(nn.Module):
    def __init__(self, block, num_blocks, num_classes=5):
        super(ResNet1D, self).__init__()
        self.in_channels = 64
        self.conv1 = nn.Conv1d(12, 64, 15, 2, 7, bias=False)
        self.bn1 = nn.BatchNorm1d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool1d(3, 2, 1)
        self.layer1 = self._make_layer(block, 64,  num_blocks[0], 1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], 2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], 2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], 2)
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, block, out_channels, num_blocks, stride):
        layers = []
        for s in [stride] + [1] * (num_blocks - 1):
            layers.append(block(self.in_channels, out_channels, s))
            self.in_channels = out_channels
        return nn.Sequential(*layers)

    def forward(self, x):
        out = self.maxpool(self.relu(self.bn1(self.conv1(x))))
        out = self.layer4(self.layer3(self.layer2(self.layer1(out))))
        return self.fc(self.flatten(self.avgpool(out)))


def ResNet18_1D():
    return ResNet1D(ResidualBlock, [2, 2, 2, 2])


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=1000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float()
                             * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:x.size(0), :]


class EcgTransformer(nn.Module):
    def __init__(self, num_classes=5, d_model=12, nhead=4, num_encoder_layers=3,
                 dim_feedforward=128, dropout=0.1):
        super(EcgTransformer, self).__init__()
        self.d_model = d_model
        self.pos_encoder = PositionalEncoding(d_model)
        layers = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward,
                                            dropout, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(layers, num_encoder_layers)
        self.conv_reducer = nn.Conv1d(12, d_model, kernel_size=1)
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(d_model * 1000, num_classes)

    def forward(self, src):
        src = self.conv_reducer(src)
        src = src.permute(0, 2, 1)
        src = self.pos_encoder(src.permute(1, 0, 2)).permute(1, 0, 2)
        out = self.transformer_encoder(src)
        return self.fc(self.flatten(out))


# =============================================================================
# VERİ
# =============================================================================
class EcgDataset(Dataset):
    def __init__(self, meta_df, labels, data_dir, is_train=False):
        self.meta_df, self.labels = meta_df, labels
        self.data_dir, self.is_train = data_dir, is_train

    def __len__(self):
        return len(self.meta_df)

    def __getitem__(self, idx):
        p = os.path.join(self.data_dir, self.meta_df.iloc[idx]['filename_lr'])
        sig, _ = wfdb.rdsamp(p)
        sig = np.transpose(sig, (1, 0))
        if self.is_train:
            sig = sig * (1 + np.random.uniform(-0.1, 0.1))
            sig = sig + np.random.normal(0, 0.01, sig.shape)
            sig = np.roll(sig, np.random.randint(-50, 50), axis=1)
        return (torch.tensor(sig, dtype=torch.float32),
                torch.tensor(self.labels[idx], dtype=torch.float32))


def _prepare(data_dir):
    df  = pd.read_csv(os.path.join(data_dir, 'ptbxl_database.csv'))
    scp = pd.read_csv(os.path.join(data_dir, 'scp_statements.csv'), index_col=0)
    dmap = scp[scp.diagnostic_class.isin(CLASS_NAMES)]['diagnostic_class'].to_dict()

    def to_super(s):
        d = ast.literal_eval(s)
        return sorted({dmap[c] for c, w in d.items() if c in dmap and w >= 50})

    df['labels'] = df.scp_codes.apply(to_super)
    tr = df[df.strat_fold.isin(range(1, 9))]
    va = df[df.strat_fold == 9]
    te = df[df.strat_fold == 10]
    mlb = MultiLabelBinarizer(classes=CLASS_NAMES)
    return (tr, mlb.fit_transform(tr['labels']),
            va, mlb.transform(va['labels']),
            te, mlb.transform(te['labels']))


# =============================================================================
# EĞİTİM / DEĞERLENDİRME
# =============================================================================
def _evaluate(model, loader, crit):
    model.eval(); L, P, tot, n = [], [], 0.0, 0
    with torch.no_grad():
        for sig, lab in loader:
            sig, lab = sig.to(device), lab.to(device)
            lo = model(sig)
            tot += crit(lo, lab).item() * sig.size(0); n += sig.size(0)
            L.append(lab.cpu().numpy()); P.append(torch.sigmoid(lo).cpu().numpy())
    L, P = np.concatenate(L), np.concatenate(P)
    return L, P, tot / n


def _train_one(name, build_fn, epochs, use_weight, use_aug, use_sched,
               data, out_dir, seed=SEED):
    tr, y_tr, va, y_va, te, y_te, data_dir = data
    print(f"\n{'='*62}\n{name}  |  {epochs} epoch  |  weight={use_weight}  "
          f"aug={use_aug}  sched={use_sched}\n{'='*62}")
    torch.manual_seed(seed); np.random.seed(seed)

    tl = DataLoader(EcgDataset(tr, y_tr, data_dir, use_aug), batch_size=32, shuffle=True)
    vl = DataLoader(EcgDataset(va, y_va, data_dir), batch_size=32)
    el = DataLoader(EcgDataset(te, y_te, data_dir), batch_size=32)

    model = build_fn().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    sched = (torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min',
             factor=0.1, patience=3) if use_sched else None)

    if use_weight:
        pos = y_tr.sum(axis=0)
        w = torch.tensor(len(y_tr) / (len(CLASS_NAMES) * pos),
                         dtype=torch.float32).to(device)
        crit = nn.BCEWithLogitsLoss(pos_weight=w)
    else:
        crit = nn.BCEWithLogitsLoss()

    ckpt = os.path.join(out_dir, f"cmp_{name.replace('+','_').replace(' ','_')}.pth")
    best, best_state = 0.0, None
    for ep in range(epochs):
        model.train()
        for sig, lab in tl:
            sig, lab = sig.to(device), lab.to(device)
            loss = crit(model(sig), lab)
            opt.zero_grad(); loss.backward(); opt.step()
        Lv, Pv, vloss = _evaluate(model, vl, crit)
        auc = roc_auc_score(Lv, Pv, average='macro')
        if sched: sched.step(vloss)
        if auc > best:
            best = auc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(f"  ep {ep+1:2d}/{epochs}  valAUC {auc:.4f}{'  *' if auc == best else ''}")

    model.load_state_dict(best_state)
    torch.save(best_state, ckpt)

    Lv, Pv, _ = _evaluate(model, vl, crit)
    Lt, Pt, _ = _evaluate(model, el, crit)
    r = {'model': name, 'epochs': epochs,
         'val_auc':  round(roc_auc_score(Lv, Pv, average='macro'), 4),
         'val_f1':   round(f1_score(Lv, (Pv > 0.5).astype(int), average='macro',
                                    zero_division=0), 3),
         'test_auc': round(roc_auc_score(Lt, Pt, average='macro'), 4),
         'test_f1':  round(f1_score(Lt, (Pt > 0.5).astype(int), average='macro',
                                    zero_division=0), 3)}
    print(f"  -> valAUC {r['val_auc']}  valF1 {r['val_f1']}  "
          f"testAUC {r['test_auc']}  testF1 {r['test_f1']}")
    return r


def compare_models(data_dir=None, out_dir=None, include_optimized=False):
    """
    include_optimized=False (varsayılan): optimized model YENİDEN EĞİTİLMEZ.
    Mevcut ecg_optimized_patientwise.pth ağırlıkları XAI analizlerinin
    dayanağı olduğu için üzerine yazılmamalı; o satırı elde mevcut
    sayılarla doldurun (val AUC 0.9338 / test AUC 0.9301 / test F1 0.73).
    """
    global DATA_DIR, OUT_DIR
    if data_dir is not None: DATA_DIR = data_dir
    if out_dir  is not None: OUT_DIR  = out_dir
    os.makedirs(OUT_DIR, exist_ok=True)

    tr, y_tr, va, y_va, te, y_te = _prepare(DATA_DIR)
    print(f"train {len(tr)}  val {len(va)}  test {len(te)}")
    data = (tr, y_tr, va, y_va, te, y_te, DATA_DIR)

    configs = [
        ("CNN",            lambda: Simple1DCNN(),   5,  False, False, False),
        ("ResNet",         ResNet18_1D,            10,  False, False, False),
        ("ResNet+weight",  ResNet18_1D,            10,  True,  False, False),
        ("ResNet+aug",     ResNet18_1D,            15,  True,  True,  False),
        ("Transformer",    lambda: EcgTransformer(), 15, True,  True,  False),
    ]
    if include_optimized:
        configs.append(("Optimized", ResNet18_1D, 20, True, True, True))

    rows = [_train_one(n, f, e, w, a, s, data, OUT_DIR)
            for n, f, e, w, a, s in configs]

    res = pd.DataFrame(rows)
    res.to_csv(os.path.join(OUT_DIR, 'model_comparison.csv'), index=False)
    print("\n=== KARŞILAŞTIRMA (hepsi patient-wise, best-checkpoint) ===")
    print(res.to_string(index=False))
    if not include_optimized:
        print("\nNOT: 'Optimized' satırı için mevcut model:")
        print("  val AUC 0.9338 | test AUC 0.9301 | test F1 0.73")
    return res


---
## 12. Figures

Grad-CAM collapse, attribution overlay, insertion/deletion curve, misclassified
recording, and the old-vs-new leakage check. Each figure cell reuses the engine
defined in Section 8.

### 12.1 Grad-CAM collapse + attribution overlay

In [ ]:
# =============================================================================
# TEZ GÖRSELLERİ v2 — DÜZELTİLMİŞ
#   Fig A: gerçekten ÇÖKEN bir NORM kaydı bul (harita = tümüyle sıfır)
#   Fig B: tek atım penceresi + okunur ısı haritası (tüm segment değil)
# =============================================================================
import os, numpy as np, torch, wfdb
import matplotlib.pyplot as plt
# xai already defined in Section 8 (inlined)

plt.rcParams.update({'font.family':'serif','font.size':11,'savefig.dpi':300,
    'savefig.bbox':'tight','figure.facecolor':'white'})
FIG='figures/'; os.makedirs(FIG, exist_ok=True)
CL=TARGET_CLASSES; LEAD=1
f10 = df[df.strat_fold==10]

def load_sig(idx):
    s,_=wfdb.rdsamp(os.path.join(data_dir, df.iloc[idx]['filename_lr']))
    return np.transpose(s,(1,0)).astype(np.float32)

# =========================================================================
# FIG A — GERÇEKTEN ÇÖKEN kayıt: ReLU haritası std==0 olan NORM
# =========================================================================
gc_relu = xai.GradCAMrelu1D(model, model.layer4[-1].conv2)
collapsed = None
for idx in f10.index:
    sig = load_sig(idx)
    cam,pred,probs = gc_relu(torch.FloatTensor(sig).unsqueeze(0).to(device))
    if CL[pred]=='NORM' and np.std(cam)==0:      # ÇÖKMÜŞ harita
        collapsed = (idx, sig, pred, cam)
        break
gc_relu.remove()

if collapsed is None:
    print("UYARI: çökmüş NORM bulunamadı; herhangi bir çökmüş kayıt aranıyor")
    gc_relu = xai.GradCAMrelu1D(model, model.layer4[-1].conv2)
    for idx in f10.index:
        sig=load_sig(idx)
        cam,pred,_=gc_relu(torch.FloatTensor(sig).unsqueeze(0).to(device))
        if np.std(cam)==0: collapsed=(idx,sig,pred,cam); break
    gc_relu.remove()

idx,sig,pred,cam_relu = collapsed
cam_signed = xai.attr_signed(model, sig, pred)
t=np.arange(1000)/100

fig,axes=plt.subplots(2,1,figsize=(9,5),sharex=True)
# üst: ReLU — çökmüş harita GERÇEKTEN beyaz kalsın (overlay çizme)
axes[0].plot(t,sig[LEAD],color='#111',lw=0.9,zorder=3)
if np.std(cam_relu)==0:
    axes[0].text(5, sig[LEAD].max()*0.75, 'attribution map $=$ 0 everywhere',
                 ha='center', fontsize=10, style='italic', color='#999')
else:
    axes[0].imshow(cam_relu[None,:],aspect='auto',cmap='Reds',alpha=0.6,
        vmin=0,vmax=1,extent=[0,10,sig[LEAD].min(),sig[LEAD].max()],zorder=1)
axes[0].set_title('Grad-CAM (with ReLU) — map is entirely zero',fontsize=11,loc='left')
axes[0].set_ylabel('Lead II (mV)')
# alt: signed — dolu harita
cs=cam_signed.copy(); cs=(cs-cs.min())/(cs.max()-cs.min()+1e-9)
axes[1].plot(t,sig[LEAD],color='#111',lw=0.9,zorder=3)
axes[1].imshow(cs[None,:],aspect='auto',cmap='Reds',alpha=0.6,
    vmin=0,vmax=1,extent=[0,10,sig[LEAD].min(),sig[LEAD].max()],zorder=1)
axes[1].set_title('Signed Grad-CAM — same recording, informative map',fontsize=11,loc='left')
axes[1].set_ylabel('Lead II (mV)'); axes[1].set_xlabel('Time (s)')
for ax in axes: ax.spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.savefig(f'{FIG}fig_gradcam_collapse.png'); plt.close()
print(f"A ok: çökmüş NORM kaydı idx={idx}, ReLU std={np.std(cam_relu):.4f} (0 olmalı)")

# =========================================================================
# FIG B — TEK ATIM penceresi + okunur overlay
#   Tüm 10 sn yerine bir QRS etrafında ~1 sn pencere; atıf o pencerede net görünür
# =========================================================================
def first_pred(name):
    for i in f10.index:
        s=load_sig(i)
        with torch.no_grad():
            p=torch.sigmoid(model(torch.FloatTensor(s).unsqueeze(0).to(device)))[0].cpu().numpy()
        if CL[int(p.argmax())]==name and p.max()>0.6: return i,s,int(p.argmax())
    return None
res = first_pred('MI') or first_pred('STTC') or first_pred('NORM')
idx,sig,cls = res

bg_idx=f10.sample(50,random_state=42).index
bg=np.stack([load_sig(i) for i in bg_idx])
expl=xai.make_shap_explainer(model, torch.FloatTensor(bg).to(device))
attrs={'Integrated Gradients':xai.attr_ig(model,sig,cls),
       'GradientSHAP':xai.attr_shap(expl,sig,cls),
       'Signed Grad-CAM':xai.attr_signed(model,sig,cls)}

# en büyük R tepesini bul, etrafında 1 sn (100 örnek) pencere
r = int(np.argmax(sig[LEAD]))
a=max(0,r-50); b=min(1000,r+70)
tw=np.arange(a,b)/100

fig,axes=plt.subplots(3,1,figsize=(8,6.5),sharex=True)
for ax,(name,at) in zip(axes,attrs.items()):
    seg=sig[LEAD][a:b]
    ax.plot(tw,seg,color='#111',lw=1.3,zorder=3)
    an=np.abs(at[a:b]); an=(an-an.min())/(an.max()-an.min()+1e-9)
    ax.imshow(an[None,:],aspect='auto',cmap='viridis',alpha=0.6,
        vmin=0,vmax=1,extent=[tw[0],tw[-1],seg.min(),seg.max()],zorder=1)
    ax.set_title(name,fontsize=11,loc='left')
    ax.set_ylabel('Lead II (mV)')
    ax.spines[['top','right']].set_visible(False)
axes[-1].set_xlabel('Time (s)')
plt.tight_layout(); plt.savefig(f'{FIG}fig_attribution_overlay.png'); plt.close()
print(f"B ok: {CL[cls]} kaydı idx={idx}, tek atım penceresi [{a},{b}]")
print("Üretilenler:", [f for f in os.listdir(FIG) if 'collapse' in f or 'overlay' in f])


### 12.2 Insertion / deletion faithfulness curve

In [ ]:
# =============================================================================
# FAITHFULNESS EĞRİSİ FİGÜRÜ (notebook'ta çalışır)
# insertion/deletion metriğinin NASIL çalıştığını gösteren örnek eğri.
# Bir MI kaydında IG vs signed Grad-CAM deletion eğrileri.
# Gereksinim: model, df, data_dir, device, TARGET_CLASSES, xai_pipeline_v3
# Çıktı: figures/fig_id_curve.png
# =============================================================================
import os, numpy as np, torch, wfdb
import matplotlib.pyplot as plt
# xai already defined in Section 8 (inlined)

plt.rcParams.update({'font.family':'serif','font.size':11,'savefig.dpi':300,
    'savefig.bbox':'tight','figure.facecolor':'white'})
FIG='figures/'; os.makedirs(FIG, exist_ok=True)
CL=TARGET_CLASSES
f10=df[df.strat_fold==10]

def load_sig(idx):
    s,_=wfdb.rdsamp(os.path.join(data_dir, df.iloc[idx]['filename_lr']))
    return np.transpose(s,(1,0)).astype(np.float32)

# güvenle doğru sınıflanan bir MI kaydı seç
def pick(name):
    for i in f10.index:
        s=load_sig(i)
        with torch.no_grad():
            p=torch.sigmoid(model(torch.FloatTensor(s).unsqueeze(0).to(device)))[0].cpu().numpy()
        if CL[int(p.argmax())]==name and p.max()>0.7: return i,s,int(p.argmax())
    return None
idx,sig,cls = pick('MI') or pick('NORM')

# insertion/deletion eğrilerini adım adım hesapla (n=50 nokta, pürüzsüz eğri)
def id_curves(attr, n=50):
    L=sig.shape[1]; base=np.median(sig,axis=1,keepdims=True)
    order=np.argsort(-np.abs(attr))
    fr=np.linspace(0,1,n+1); base_full=np.repeat(base,L,axis=1)
    ins,dele=[],[]
    for f in fr:
        k=int(f*L); top=order[:k]
        d=sig.copy(); s=base_full.copy()
        if k: d[:,top]=base; s[:,top]=sig[:,top]
        dele.append(xai.model_prob(model,d,cls))
        ins.append(xai.model_prob(model,s,cls))
    return fr,np.array(ins),np.array(dele)

attrs={'Integrated Gradients':xai.attr_ig(model,sig,cls),
       'Signed Grad-CAM':xai.attr_signed(model,sig,cls)}
colors={'Integrated Gradients':'#4C72B0','Signed Grad-CAM':'#C44E52'}

fig,axes=plt.subplots(1,2,figsize=(10,4))
for name,a in attrs.items():
    fr,ins,dele=id_curves(a)
    axes[0].plot(fr,ins,label=name,color=colors[name],lw=1.8)
    axes[1].plot(fr,dele,label=name,color=colors[name],lw=1.8)
axes[0].set_title('Insertion (higher = more faithful)',fontsize=11)
axes[1].set_title('Deletion (lower = more faithful)',fontsize=11)
for ax in axes:
    ax.set_xlabel('Fraction of timepoints'); ax.set_ylabel('Predicted probability')
    ax.set_ylim(-0.02,1.02); ax.legend(frameon=False,fontsize=9)
    ax.spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.savefig(f'{FIG}fig_id_curve.png'); plt.close()
print(f"ok: fig_id_curve.png  ({CL[cls]} kaydı idx={idx})")


### 12.3 Misclassified recording

In [ ]:
# =============================================================================
# MISCLASSIFIED ÖRNEK — yanlış sınıflandırılan bir kayıtta XAI
# =============================================================================
# Hoca: analiz sadece doğru sınıflandırılanlarda; bir de yanlış olanı göster.
# Bu hücre: modelin YANLIŞ tahmin ettiği bir kayıt bulur, IG haritasını
#   (a) tahmin edilen (yanlış) sınıf için ve (b) gerçek sınıf için çizer.
# Gereksinim: model, df, data_dir, device, TARGET_CLASSES, xai_pipeline_v3
# Çıktı: figures/fig_misclassified.png
# =============================================================================
import os, numpy as np, torch, wfdb
import matplotlib.pyplot as plt
# xai already defined in Section 8 (inlined)

plt.rcParams.update({'font.family':'serif','font.size':11,'savefig.dpi':300,
    'savefig.bbox':'tight','figure.facecolor':'white'})
FIG='figures/'; os.makedirs(FIG, exist_ok=True)
CL=TARGET_CLASSES; LEAD=1
f10=df[df.strat_fold==10]

def load_sig(idx):
    s,_=wfdb.rdsamp(os.path.join(data_dir, df.loc[idx,'filename_lr']))
    return np.transpose(s,(1,0)).astype(np.float32)

# gerçek etiketleri pipeline'ın kendi fonksiyonundan al (scp_codes -> superclass)
f10 = xai._fold10_labels(df)          # 'labels' kolonu ekli fold-10
print("f10 satır:", len(f10), "| örnek etiket:", f10['labels'].iloc[0])

def true_label(idx):
    labs = f10.loc[idx,'labels'] if idx in f10.index else []
    # tek-etiketli kayıtları tercih et; birden fazla varsa ilkini al
    for c in CL:
        if c in labs:
            return c
    return None

# YANLIŞ sınıflandırılan, güvenli (yüksek olasılıklı yanlış) bir kayıt bul
# tercih: MI'ı STTC sanılan gibi klinik açıdan ilginç bir hata
found=None; fallback=None
for idx in f10.index:
    tl=true_label(idx)
    if tl is None: continue
    sig=load_sig(idx)
    with torch.no_grad():
        p=torch.sigmoid(model(torch.FloatTensor(sig).unsqueeze(0).to(device)))[0].cpu().numpy()
    pred=CL[int(p.argmax())]
    if pred!=tl and p.max()>0.6:          # kendinden emin ama YANLIŞ
        if fallback is None: fallback=(idx,sig,tl,pred,p)  # ilk yanlışı sakla
        if tl=='MI':                        # MI hatası tercih (klinik ilginç)
            found=(idx,sig,tl,pred,p); break
if found is None: found=fallback
assert found is not None, "Yüksek güvenli yanlış kayıt bulunamadı; eşiği düşür (0.6 -> 0.5)"
idx,sig,tl,pred,probs=found
print(f"Yanlış kayıt idx={idx}: gerçek={tl}, tahmin={pred} (p={probs.max():.2f})")

# IG haritası: hem tahmin edilen (yanlış) sınıf hem gerçek sınıf için
attr_pred = xai.attr_ig(model, sig, CL.index(pred))
attr_true = xai.attr_ig(model, sig, CL.index(tl))
t=np.arange(1000)/100

fig,axes=plt.subplots(2,1,figsize=(9,5),sharex=True)
for ax,attr,title in [
    (axes[0],attr_pred,f'IG for predicted class "{pred}" (model\'s wrong decision)'),
    (axes[1],attr_true,f'IG for true class "{tl}" (correct label)')]:
    ax.plot(t,sig[LEAD],color='#111',lw=0.9,zorder=3)
    a=np.abs(attr); a=(a-a.min())/(a.max()-a.min()+1e-9)
    ax.imshow(a[None,:],aspect='auto',cmap='viridis',alpha=0.55,vmin=0,vmax=1,
              extent=[0,10,sig[LEAD].min(),sig[LEAD].max()],zorder=1)
    ax.set_title(title,fontsize=10.5,loc='left')
    ax.set_ylabel('Lead II (mV)')
    ax.spines[['top','right']].set_visible(False)
axes[1].set_xlabel('Time (s)')
plt.tight_layout(); plt.savefig(f'{FIG}fig_misclassified.png'); plt.close()
print("fig_misclassified.png üretildi")

# faithfulness'ı bu yanlış kayıtta da ölç (doğru olanlarla kıyas için)
ins_p,del_p=xai.insertion_deletion(model,sig,attr_pred,CL.index(pred))
ins_t,del_t=xai.insertion_deletion(model,sig,attr_true,CL.index(tl))
print(f"Yanlış kayıtta IG faithfulness:")
print(f"  predicted class '{pred}': {ins_p-del_p:.3f}  (ins={ins_p:.3f}, del={del_p:.3f})")
print(f"  true class      '{tl}':   {ins_t-del_t:.3f}  (ins={ins_t:.3f}, del={del_t:.3f})")


### 12.4 Old vs new model — data-leakage check

In [ ]:
# =============================================================================
# ESKİ vs YENİ MODEL — anket EKG'lerinde Grad-CAM karşılaştırması
# =============================================================================
# Amaç: ankette doktora gösterilen haritaların (eski, sızıntılı model) tezde
# değerlendirilen modelin (yeni, patient-wise) haritalarından NE KADAR farklı
# olduğunu ölçmek. Aynı EKG, iki model, yan yana.
#
# Gereksinim (kernelde hazır): model (YENİ), df, data_dir, device,
#   TARGET_CLASSES, xai_pipeline_v3
# Ek: eski model ağırlığı ./ENK/TEZ/ecg_optimized_model.pth (veya yol düzelt)
#
# Üretir:
#   figures/fig_model_compare.png   : 4 örnek EKG, eski vs yeni harita
#   konsol: anket EKG'leri için harita benzerliği (Spearman, top-k Jaccard)
# =============================================================================
import os, numpy as np, torch, wfdb
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
# xai already defined in Section 8 (inlined)

plt.rcParams.update({'font.family':'serif','font.size':10,'savefig.dpi':300,
    'savefig.bbox':'tight','figure.facecolor':'white'})
FIG='figures/'; os.makedirs(FIG, exist_ok=True)
CL=TARGET_CLASSES; LEAD=1
OLD_PATH='./ENK/TEZ/ecg_optimized_model.pth'   # <-- gerekirse yolu düzelt

# 20 anket EKG'si (df satır indeksi)
SURVEY=[(1,'NORM',2927),(2,'NORM',14133),(3,'NORM',7769),(4,'NORM',15678),
        (5,'MI',955),(6,'MI',13810),(7,'MI',4155),(8,'MI',3107),
        (9,'STTC',11105),(10,'STTC',18386),(11,'STTC',1611),(12,'STTC',17102),
        (13,'CD',1207),(14,'CD',17501),(15,'CD',15064),(16,'CD',1043),
        (17,'HYP',15749),(18,'HYP',1772),(19,'HYP',12160),(20,'HYP',9000)]

def load_sig(idx):
    s,_=wfdb.rdsamp(os.path.join(data_dir, df.iloc[idx]['filename_lr']))
    return np.transpose(s,(1,0)).astype(np.float32)

# --- eski modeli yükle (yeni model zaten 'model' değişkeninde) ---
old_model = xai.ResNet18_1D().to(device)
old_model.load_state_dict(torch.load(OLD_PATH, map_location=device))
old_model.eval()
new_model = model
print("İki model yüklendi (eski + yeni)")

# --- her anket EKG'si için iki modelden signed Grad-CAM ---
def gradcam_for(m, sig, cls):
    gc = xai.GradCAMsigned1D(m, m.layer4[-1].conv2)
    cam,_,_ = gc(torch.FloatTensor(sig).unsqueeze(0).to(device), class_idx=cls)
    gc.remove()
    return cam

def topk_jaccard(a,b,k=50):
    ta=set(np.argsort(-np.abs(a))[:k]); tb=set(np.argsort(-np.abs(b))[:k])
    return len(ta&tb)/len(ta|tb)

rows=[]
cams_old, cams_new = {}, {}
for n,true_cls,idx in SURVEY:
    sig=load_sig(idx)
    cls=CL.index(true_cls)         # gerçek sınıf hedefli (karşılaştırılabilir)
    co=gradcam_for(old_model,sig,cls)
    cn=gradcam_for(new_model,sig,cls)
    cams_old[n]=(sig,co); cams_new[n]=(sig,cn)
    rho=spearmanr(co,cn).correlation
    jac=topk_jaccard(co,cn)
    rows.append((n,true_cls,rho,jac))
    print(f"  #{n:2d} {true_cls:4s}  Spearman={rho:+.3f}  top50-Jaccard={jac:.3f}")

rhos=[r[2] for r in rows if not np.isnan(r[2])]
jacs=[r[3] for r in rows]
print(f"\nOrtalama: Spearman={np.mean(rhos):.3f}  Jaccard={np.mean(jacs):.3f}")
print("(Düşük değer = iki modelin haritaları farklı = sınırlama gerçek)")
print("(Yüksek değer = haritalar benzer = sınırlama küçük)")

# --- görsel: 4 sınıftan birer örnek, eski vs yeni yan yana ---
examples=[1,5,9,17]   # NORM, MI, STTC, HYP
fig,axes=plt.subplots(len(examples),2,figsize=(11,2.4*len(examples)),sharex=True)
for row,n in enumerate(examples):
    for col,(cams,title) in enumerate([(cams_old,'Old model (survey)'),
                                        (cams_new,'New model (thesis)')]):
        sig,cam=cams[n]; ax=axes[row,col]
        t=np.arange(1000)/100
        ax.plot(t,sig[LEAD],color='#111',lw=0.8,zorder=3)
        c=(cam-cam.min())/(cam.max()-cam.min()+1e-9)
        ax.imshow(c[None,:],aspect='auto',cmap='Reds',alpha=0.55,vmin=0,vmax=1,
                  extent=[0,10,sig[LEAD].min(),sig[LEAD].max()],zorder=1)
        cls_name=[s for s in SURVEY if s[0]==n][0][1]
        if row==0: ax.set_title(title,fontsize=11)
        if col==0: ax.set_ylabel(f'{cls_name}\nLead II (mV)',fontsize=9)
        ax.spines[['top','right']].set_visible(False)
axes[-1,0].set_xlabel('Time (s)'); axes[-1,1].set_xlabel('Time (s)')
plt.tight_layout(); plt.savefig(f'{FIG}fig_model_compare.png'); plt.close()
print(f"\nfig_model_compare.png üretildi ({len(examples)} örnek)")


In [ ]:
# =============================================================================
# ESKİ vs YENİ MODEL — 20 anket EKG'sinde TANISAL DOĞRULUK
# Soru: sızıntılı eski model, anket EKG'lerinde yeni modelden farklı mı
#       tahmin ediyordu? (doktora gösterilen doğruluk değişti mi?)
# Gereksinim: model (YENİ), df, data_dir, device, TARGET_CLASSES, xai_pipeline_v3
# =============================================================================
import os, numpy as np, torch, wfdb, pandas as pd
# xai already defined in Section 8 (inlined)

CL=TARGET_CLASSES
OLD_PATH='./ENK/TEZ/ecg_optimized_model.pth'   # gerekirse düzelt

SURVEY=[(1,'NORM',2927),(2,'NORM',14133),(3,'NORM',7769),(4,'NORM',15678),
        (5,'MI',955),(6,'MI',13810),(7,'MI',4155),(8,'MI',3107),
        (9,'STTC',11105),(10,'STTC',18386),(11,'STTC',1611),(12,'STTC',17102),
        (13,'CD',1207),(14,'CD',17501),(15,'CD',15064),(16,'CD',1043),
        (17,'HYP',15749),(18,'HYP',1772),(19,'HYP',12160),(20,'HYP',9000)]

def load_sig(idx):
    s,_=wfdb.rdsamp(os.path.join(data_dir, df.iloc[idx]['filename_lr']))
    return np.transpose(s,(1,0)).astype(np.float32)

old_model=xai.ResNet18_1D().to(device)
old_model.load_state_dict(torch.load(OLD_PATH, map_location=device))
old_model.eval()

def predict(m,sig):
    with torch.no_grad():
        p=torch.sigmoid(m(torch.FloatTensor(sig).unsqueeze(0).to(device)))[0].cpu().numpy()
    return CL[int(p.argmax())], float(p.max())

rows=[]
for n,true_cls,idx in SURVEY:
    sig=load_sig(idx)
    op,opb=predict(old_model,sig)
    np_,npb=predict(model,sig)
    rows.append({'ecg':n,'true':true_cls,
                 'old_pred':op,'old_ok':op==true_cls,
                 'new_pred':np_,'new_ok':np_==true_cls})
d=pd.DataFrame(rows)
print(d.to_string(index=False))
print()
print(f"ESKİ model doğru: {d.old_ok.sum()}/20 = {d.old_ok.mean():.0%}")
print(f"YENİ model doğru: {d.new_ok.sum()}/20 = {d.new_ok.mean():.0%}")
print()
# sınıf bazında
print("Sınıf bazında (eski / yeni doğru sayısı):")
for c in CL:
    sub=d[d.true==c]
    print(f"  {c:4s}: eski {sub.old_ok.sum()}/4   yeni {sub.new_ok.sum()}/4")
print()
# tahmini değişen EKG'ler
changed=d[d.old_pred!=d.new_pred]
print(f"Tahmini değişen EKG sayısı: {len(changed)}/20")
if len(changed):
    print(changed[['ecg','true','old_pred','new_pred']].to_string(index=False))
